# mx3sw tutorial

This notebook shows the two most common analysis steps for MuMax3 spin-wave simulations:

1. **FMR frequency** from `table.txt`, and why the uneven save times of MuMax3 matter.
2. **Dispersion relation f(k)** from a space-time map of the magnetisation.

It uses **synthetic data** that behaves like MuMax3 output, so it runs anywhere in a few seconds, without MuMax3 or a GPU.
With your own simulation, replace the data-generation cells with `mx3sw.read_table(...)` and `mx3sw.read_ovf_series(...)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mx3sw

plt.rcParams.update({"figure.dpi": 100, "svg.fonttype": "none", "font.size": 10})
rng = np.random.default_rng(0)

## 1. FMR frequency from `table.txt`

MuMax3's adaptive solver writes `table.txt` rows at **slightly uneven** times, even with `TableAutoSave(dt)`.
Below we make a table like that: a damped oscillation at a known 7.60 GHz. Most rows are ~27 ps apart, but the solver sometimes shortens a step to land on a save point, so some gaps are ~15 ps. (A real MuMax3 table showed the same pattern: median spacing 27 ps for `TableAutoSave(25e-12)`.)

In [2]:
f_true = 7.60e9
steps = np.where(rng.random(800) < 0.2, 15e-12, 27.5e-12) * rng.uniform(0.97, 1.03, 800)
t = np.cumsum(steps)                                    # uneven save times, like MuMax3
mx = 0.01 * np.exp(-t / 5e-9) * np.sin(2 * np.pi * f_true * t)

# write and read back a MuMax3-style table.txt
with open("table.txt", "w") as fh:
    fh.write("# t (s)\tmx ()\n")
    for ti, mi in zip(t, mx):
        fh.write(f"{ti}\t{mi}\n")
table = mx3sw.read_table("table.txt")
table.head()

Assuming a fixed `dt` (the median spacing) shifts the peak. Passing the real times with `times=` resamples onto an even grid first and recovers the right frequency.

In [3]:
dt_median = table["t"].diff().median()
f_naive = mx3sw.peak_frequency(table["mx"], dt_median)
f_fixed = mx3sw.peak_frequency(table["mx"], times=table["t"])
print(f"true      : {f_true / 1e9:.2f} GHz")
print(f"fixed dt  : {f_naive / 1e9:.2f} GHz   (error {abs(f_naive - f_true) / f_true:.1%})")
print(f"times=    : {f_fixed / 1e9:.2f} GHz")

true      : 7.60 GHz
fixed dt  : 6.96 GHz   (error 8.4%)
times=    : 7.59 GHz


In [4]:
f1, a1 = mx3sw.fft_spectrum(table["mx"], dt_median)
f2, a2 = mx3sw.fft_spectrum(table["mx"], times=table["t"])

fig, ax = plt.subplots(figsize=(6, 3.2))
w1, w2 = (f1 > 4e9) & (f1 < 11e9), (f2 > 4e9) & (f2 < 11e9)
ax.plot(f1[w1] / 1e9, a1[w1] / a1.max(), color="0.6", lw=1.2, label="fixed dt (wrong)")
ax.plot(f2[w2] / 1e9, a2[w2] / a2.max(), color="C0", lw=1.6, label="times= (resampled)")
ax.axvline(f_true / 1e9, color="C3", ls="--", lw=1, label="true 7.60 GHz")
ax.set_xlim(4, 11)
ax.set_xlabel("Frequency (GHz)")
ax.set_ylabel("Amplitude (norm.)")
ax.set_title("FMR peak from table.txt")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("../docs/images/fmr_spectrum.svg", metadata={"Date": None})
plt.close(fig)

![FMR spectrum](https://raw.githubusercontent.com/Dark-atoms/mumax3-spinwave-tools/main/docs/images/fmr_spectrum.svg)

## 2. Dispersion relation from m(t, x)

In a real simulation you would save m along the waveguide, e.g. `AutoSave(CropY(m, ...), dt)`, then

```python
files, m = mx3sw.read_ovf_series("sim.out", "m_*.ovf")   # (time, 3, z, y, x)
mx_tx = m[:, 0, 0, 0, :]
```

Here we build `mx_tx` from many spin waves that follow the exchange dispersion of Permalloy at B = 0.1 T, plus noise.

In [5]:
Ms, Aex, B = 800e3, 13e-12, 0.1
dx, dt = 5e-9, 25e-12
x = np.arange(512) * dx
tt = np.arange(1024) * dt

ks = np.linspace(5e6, 1.1e8, 400)
fs = mx3sw.exchange_dispersion(ks, B, Aex, Ms)
phases = rng.uniform(0, 2 * np.pi, ks.size)
mx_tx = sum(np.cos(k * x[None, :] - 2 * np.pi * f * tt[:, None] + p)
            for k, f, p in zip(ks, fs, phases))
mx_tx = 1e-3 * mx_tx / ks.size + 2e-5 * rng.normal(size=mx_tx.shape)
mx_tx.shape

(1024, 512)

In [6]:
k, f, P = mx3sw.dispersion_map(mx_tx, dt, dx)

# ridge: for each positive frequency, the k with the most power
pos_k, pos_f = k > 0, f > 0
Pk = P[np.ix_(pos_f, pos_k)]
f_r = f[pos_f]
k_r = k[pos_k][np.argmax(Pk, axis=1)]
band = (f_r > fs.min()) & (f_r < fs.max())
strong = band & (Pk.max(axis=1) > 0.05 * Pk.max())
f_r, k_r = f_r[strong], k_r[strong]
print(f"{strong.sum()} ridge points")

197 ridge points


In [7]:
kk = np.linspace(0, 1.15e8, 60)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(kk / 1e6, mx3sw.exchange_dispersion(kk, B, Aex, Ms) / 1e9, color="C3", lw=1.6,
        label="theory: exchange dispersion")
step = max(1, len(k_r) // 35)
ax.plot(k_r[::step] / 1e6, f_r[::step] / 1e9, "o", ms=3.5, color="C0", label="2D FFT of m(t, x)")
ax.set_xlabel("k (rad/µm)")
ax.set_ylabel("f (GHz)")
ax.set_title("Spin-wave dispersion, Permalloy, B = 0.1 T")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("../docs/images/dispersion.svg", metadata={"Date": None})
plt.close(fig)

![Dispersion](https://raw.githubusercontent.com/Dark-atoms/mumax3-spinwave-tools/main/docs/images/dispersion.svg)

The FFT ridge follows the analytical exchange dispersion f = γ/2π · (B + 2A k²/Mₛ).

**Next steps with your own data**
- `mx3sw.read_ovf(...)` for single snapshots, `mx3sw.wavelength_from_profile(...)` for wavelengths along a line.
- `mx3sw.fit_kittel_ms(B, f)` to extract Mₛ from an FMR field sweep (see `examples/02_kittel_sweep.py`).